# Inspecting Feature Distributions Before Modeling

**Before you train any machine learning model, you must understand what your data actually looks like — not what you assume, not what the dataset description claims, but what it truly looks like when inspected column by column.**

## Overview

Feature distribution inspection is the process of examining how values in each feature are distributed — their range, central tendency, spread, skewness, frequency patterns, and anomalies. This step is often rushed or skipped by beginners who are eager to train models quickly. That is a mistake.

**Models learn patterns from data.** If your features contain extreme skew, heavy outliers, incorrect encodings, unexpected categories, or data entry errors, the model will learn those patterns too. You cannot fix what you do not inspect.

### By the end of this lesson, you will be able to:

- Systematically examine numerical feature distributions (range, skewness, outliers)
- Identify patterns and anomalies in categorical features
- Detect hidden data quality issues through distribution analysis
- Recognize when feature transformations are necessary
- Document distribution insights for reproducibility
- Make informed preprocessing decisions grounded in evidence, not habit

**Understanding feature distributions is not optional preparation. It is foundational risk management for your ML pipeline.**

## Why Feature Distribution Inspection Matters

Machine learning algorithms make assumptions about data — some explicit, some implicit.

- **Linear models** assume approximate linearity and are sensitive to outliers
- **Distance-based models** (like KNN) are sensitive to scale
- **Tree-based models** are more robust to skew but can still be influenced by extreme values
- **Neural networks** benefit from normalized inputs

### If you do not inspect distributions, you risk:

- Feeding highly skewed variables into linear models without transformation
- Allowing extreme outliers to dominate loss functions
- Treating ordinal categories as nominal or vice versa
- Ignoring rare classes that distort evaluation metrics
- Scaling variables that should not be scaled (or vice versa)
- Missing data quality issues entirely until the model fails in production

### Inspection reduces uncertainty

It allows preprocessing decisions to be **informed rather than reactive**. Instead of applying transformations because "that's what people do," you apply them because your data distribution demands it.

## Inspecting Numerical Feature Distributions

Numerical features require examination of:

- **Minimum and maximum values** — reveal range and potential extremes
- **Mean and median** — indicate central tendency and symmetry
- **Standard deviation** — quantifies dispersion
- **Percentiles** (especially 25th, 50th, 75th, 95th, 99th) — show spread and tail behavior
- **Presence of extreme values** — outliers that may distort learning
- **Skewness** — measure of asymmetry

### Summary Statistics: The First Step

Begin with summary statistics:

```python
df.describe()
```

This gives a quick overview of central tendency and dispersion. However, **summary statistics alone are insufficient**. Two features can have identical means and standard deviations but completely different distributions.

### Histograms: Visualizing Distribution Shape

Histograms help visualize how values are distributed across ranges:

```python
import matplotlib.pyplot as plt

df['MonthlyCharges'].hist(bins=30)
plt.title("Distribution of Monthly Charges")
plt.show()
```

Look for:

- **Symmetry vs skewness** — is the distribution balanced or lopsided?
- **Long tails** — indication of extreme values
- **Clustering** — do values group at certain ranges?
- **Gaps in values** — unusual patterns or missing data
- **Multiple peaks** — possible subgroups within the data

Highly right-skewed distributions (common in income, spending, or count data) may benefit from log transformation.

### Boxplots: Highlighting Outliers

Boxplots quickly identify outliers:

```python
df.boxplot(column='TotalCharges')
plt.show()
```

Boxplots show:

- **Median** (middle line)
- **Interquartile range (IQR)** (the box)
- **Outliers** beyond 1.5 × IQR (plotted as points)

Extreme outliers may distort model training. Inspection allows you to decide whether to:

- Keep them (if they represent valid real-world phenomena)
- Cap them (winsorization)
- Transform them (log, square root)
- Remove them (with justification)

### Understanding Skewness

Skewness measures asymmetry in a distribution:

- **Symmetric distribution** → skewness near 0
- **Right-skewed** (long right tail) → positive skew (common: income, spending)
- **Left-skewed** (long left tail) → negative skew

```python
df['TotalCharges'].skew()
```

**General skewness interpretation:**
- `|skew| < 0.5` → approximately symmetric
- `0.5 ≤ |skew| < 1` → moderately skewed
- `|skew| ≥ 1` → highly skewed

Highly skewed variables may cause linear models to overweight large values. Transformations such as:

- **Log transformation** — `np.log1p(x)` stabilizes exponential-like growth
- **Square root transformation** — `np.sqrt(x)` gentler than log
- **Box-Cox transformation** — automatically finds optimal transformation

These may stabilize variance and improve performance. **Inspection tells you whether transformation is necessary.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Inspect numerical features
# (Using synthetic data for demonstration)

# Create sample customer data
np.random.seed(42)
n_samples = 1000

df_demo = pd.DataFrame({
    'tenure': np.random.exponential(scale=20, size=n_samples),
    'MonthlyCharges': np.random.gamma(shape=2, scale=30, size=n_samples),
    'TotalCharges': np.random.exponential(scale=1000, size=n_samples)
})

# Step 1: Summary statistics
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(df_demo.describe())
print()

# Step 2: Skewness analysis
print("=" * 60)
print("SKEWNESS ANALYSIS")
print("=" * 60)
for col in df_demo.columns:
    skew = df_demo[col].skew()
    print(f"{col}: skewness = {skew:.3f}")
print()

# Step 3: Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, col in enumerate(df_demo.columns):
    axes[idx].hist(df_demo[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f"Distribution of {col}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# Step 4: Boxplots for outlier detection
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, col in enumerate(df_demo.columns):
    axes[idx].boxplot(df_demo[col])
    axes[idx].set_title(f"Boxplot of {col}")
    axes[idx].set_ylabel(col)

plt.tight_layout()
plt.show()

print("Key observation: Right-skewed distributions may benefit from log transformation.")

## Inspecting Categorical Feature Distributions

Categorical features must be examined for:

- **Number of unique categories** — affects encoding and model complexity
- **Category frequency distribution** — balance or severe imbalance?
- **Rare levels** — categories with very few samples
- **Inconsistent labeling** — e.g., "Yes", "yes", "YES" should be one category

### Value Counts: The Starting Point

```python
df['ContractType'].value_counts()
```

Look for:

- **Severe imbalance** — one category dominates
- **Rare categories** with very few samples (e.g., only 5 instances of a category)
- **Unexpected values** — categories that shouldn't exist
- **Data entry inconsistencies** — similar but differently spelled values

### Why Imbalance Matters

Rare categories may:

- Cause **unstable model behavior** (high variance in predictions)
- Lead to **overfitting** on small samples
- **Inflate dimensionality** after one-hot encoding (more columns to learn)
- Distort **evaluation metrics** if not handled properly

### Remediation Strategies

You may need to:

- **Group rare categories** into "Other" or a catch-all category
- **Drop extremely sparse categories** if they are truly unimportant
- **Clean inconsistent labels** (standardize case, spelling, formatting)

**Inspection prevents silent encoding problems** that manifest as poor model performance.

In [ ]:
# Example: Inspect categorical features

# Add categorical columns to demo data
df_demo['Contract'] = np.random.choice(
    ['Month-to-month', 'One year', 'Two year'],
    size=n_samples,
    p=[0.55, 0.25, 0.20]
)

df_demo['PaymentMethod'] = np.random.choice(
    ['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card', 'Other'],
    size=n_samples,
    p=[0.35, 0.20, 0.25, 0.15, 0.05]
)

# Step 1: Value counts
print("=" * 60)
print("CATEGORICAL FEATURE ANALYSIS")
print("=" * 60)

for col in ['Contract', 'PaymentMethod']:
    print(f"\n{col}:")
    print(df_demo[col].value_counts())
    print(f"Unique categories: {df_demo[col].nunique()}")
    print()

# Step 2: Identify rare categories (e.g., < 1% of data)
print("=" * 60)
print("RARE CATEGORIES (< 1% of data)")
print("=" * 60)

threshold = n_samples * 0.01
for col in ['Contract', 'PaymentMethod']:
    rare_cats = df_demo[col].value_counts()[df_demo[col].value_counts() < threshold]
    if len(rare_cats) > 0:
        print(f"\n{col}: {list(rare_cats.index)}")
    else:
        print(f"\n{col}: No rare categories")

# Step 3: Visualize categorical distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, col in enumerate(['Contract', 'PaymentMethod']):
    df_demo[col].value_counts().plot(kind='barh', ax=axes[idx])
    axes[idx].set_title(f"Distribution of {col}")
    axes[idx].set_xlabel("Count")

plt.tight_layout()
plt.show()

print("\nKey observation: PaymentMethod is imbalanced. 'Other' (5%) could be grouped with 'Mailed check'.")

## Detecting Hidden Data Quality Issues

Distribution inspection often reveals deeper issues that impact model performance:

### Impossible Values

Real-world data often contains nonsensical values that indicate corruption:

**Examples:**
- `Age = -5` (negative age)
- `MonthlyCharges = 0` when business rules say minimum is 10
- `Tenure = 500 years` (impossible tenure)
- `Temperature = 1000 Celsius` (physically implausible)

These values may indicate:
- Data entry errors
- Sensor malfunctions
- Missing value encoding mistakes (e.g., using -1 or 999 instead of NaN)

**Action:** Remove or flag these records, investigate root cause.

### Excessive Zeros

If a feature contains 95% zeros and 5% non-zero values, it may behave like a binary feature rather than continuous.

This affects:
- **Scaling decisions** (should it be scaled?)
- **Model choice** (some models assume continuous distributions)
- **Interpretation** (is zero meaningful or missing?)

**Action:** Investigate whether zeros are valid or represent missing/structural data.

### Multi-modal Distributions

Multiple peaks in a distribution may indicate:

- **Different customer segments** — age shows peaks at 20s and 60s (young and retired)
- **Mixed populations** — salary shows two peaks (entry-level and experienced)
- **Data aggregation errors** — incorrect merges or joins
- **Temporal effects** — seasonality or trend artifacts

**Recognition and action:** Recognizing this early may influence:
- Feature engineering (create segment indicators)
- Modeling strategy (train separate models per segment)
- Data quality assessment (investigate root cause)

## Comparing Feature Distributions by Target

Feature inspection becomes more powerful when you compare distributions across target classes.

This reveals **discriminative power** — can the feature actually distinguish between classes?

### Example for Binary Classification

```python
df.groupby('Churn')['MonthlyCharges'].describe()
```

Or visualize side-by-side:

```python
import seaborn as sns
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
```

### Key Questions

- Do churned customers have **higher monthly charges**?
- Do fraud cases show **different transaction amounts**?
- Does price distribution **differ by default status**?
- Are the distributions **clearly separated** or do they **overlap heavily**?

### Interpretation

- **Heavy overlap** → feature may have **weak predictive signal**
- **Clear separation** → feature may be **valuable** for discrimination

**Inspection here provides intuition about feature importance before modeling.** You might decide to:

- Drop weakly discriminative features
- Engineer new features that better separate classes
- Combine features to improve separation

In [ ]:
# Example: Compare feature distributions by target

# Create binary target (Churn)
df_demo['Churn'] = np.random.choice([0, 1], size=n_samples, p=[0.73, 0.27])

# Compare MonthlyCharges by Churn status
print("=" * 60)
print("FEATURE DISTRIBUTIONS BY TARGET CLASS")
print("=" * 60)

print("\nMonthlyCharges by Churn:")
print(df_demo.groupby('Churn')['MonthlyCharges'].describe())

print("\nTenure by Churn:")
print(df_demo.groupby('Churn')['tenure'].describe())

# Visualize with boxplots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x='Churn', y='MonthlyCharges', data=df_demo, ax=axes[0])
axes[0].set_title("MonthlyCharges by Churn")
axes[0].set_xlabel("Churn (0=No, 1=Yes)")

sns.boxplot(x='Churn', y='tenure', data=df_demo, ax=axes[1])
axes[1].set_title("Tenure by Churn")
axes[1].set_xlabel("Churn (0=No, 1=Yes)")

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Churned customers likely have higher MonthlyCharges (higher median)")
print("- Churned customers have significantly lower tenure (left shift)")
print("- Both features show discriminative power for predicting churn")

## Feature Scaling Considerations

Inspection also informs scaling decisions, which are critical for certain algorithms.

### The Scale Problem

If one feature ranges from 0 to 1 and another ranges from 0 to 100,000, distance-based models will prioritize the larger-scale feature:

```python
# Feature 1: salary (0 to 200,000)
# Feature 2: age (0 to 100)
# A distance of 10,000 in salary is tiny relative to range
# A distance of 10 in age is substantial
# But distance-based models weight salary much more heavily!
```

### Checking Ranges

```python
df[['Feature1', 'Feature2']].agg(['min', 'max'])
```

### When Scaling is Necessary

**Required scaling:**
- KNN (distance-sensitive)
- SVM (distance-sensitive)
- Neural networks (helps convergence)
- Regularized linear models (regularization term is sensitive to feature scale)

**Optional/less critical:**
- Tree-based models (less sensitive to scale)

**Important:** Even for tree-based models, inspection helps detect anomalies. A feature with extreme range outliers may still need attention.

### Scaling Methods

- **Standardization** (z-score): `(x - mean) / std` → mean=0, std=1
- **Normalization** (min-max): `(x - min) / (max - min)` → range [0,1]
- **Robust scaling**: uses median and IQR, resistant to outliers

**Choose based on distribution shape and outlier presence.**

## When to Transform Features

Inspection helps you decide **intentionally** whether to apply transformations. This is not habit — it is evidence-based.

### Log Transformation

**When to apply:**
- Highly right-skewed (skewness > 1)
- Exponential-like growth pattern
- Values span multiple orders of magnitude (e.g., 1, 10, 100, 1000)

**Examples:** income, spending, population, count data

```python
import numpy as np
df['Income_log'] = np.log1p(df['Income'])  # log1p handles zeros
```

Then re-plot histogram to confirm the transformation improved symmetry.

### Square Root Transformation

**When to apply:**
- Moderately right-skewed (0.5 < skewness < 1)
- Gentler than log transformation
- Count data with moderate skew

```python
df['Count_sqrt'] = np.sqrt(df['Count'])
```

### Box-Cox Transformation

**When to apply:**
- Automatic optimization of transformation parameter
- When you're unsure which transformation is best
- All values must be positive

```python
from scipy.stats import boxcox
df['Feature_boxcox'], lambda_param = boxcox(df['Feature'])
```

### Why Transformation Decisions Must Be Justified

Transformations:
- **Reduce interpretability** (what does log(income) mean to stakeholders?)
- **Must be applied consistently** to training and test data
- **Should improve model performance or stability**, not just for aesthetic reasons

In [ ]:
# Example: Demonstrating when to transform features

# Create highly skewed feature (exponential distribution)
df_demo['Revenue'] = np.random.exponential(scale=500, size=n_samples)

# Calculate skewness before transformation
skew_original = df_demo['Revenue'].skew()
print(f"Original Revenue skewness: {skew_original:.3f}")

# Apply log transformation
df_demo['Revenue_log'] = np.log1p(df_demo['Revenue'])
skew_transformed = df_demo['Revenue_log'].skew()
print(f"Log-transformed Revenue skewness: {skew_transformed:.3f}")

# Visualize before and after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_demo['Revenue'], bins=40, edgecolor='black', alpha=0.7)
axes[0].set_title(f"Original Revenue (skewness={skew_original:.3f})")
axes[0].set_xlabel("Revenue")
axes[0].set_ylabel("Frequency")

axes[1].hist(df_demo['Revenue_log'], bins=40, edgecolor='black', alpha=0.7)
axes[1].set_title(f"Log-Transformed Revenue (skewness={skew_transformed:.3f})")
axes[1].set_xlabel("log(Revenue)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print("\nKey insight: Log transformation successfully reduced skewness.")
print("This would be beneficial for linear models or models sensitive to skew.")

## Systematic Inspection Workflow

Before modeling, follow this structured approach to ensure nothing is overlooked:

### Step 1: Load and Characterize
- Load dataset
- Print shape (`df.shape`) and data types (`df.dtypes`)
- Check for missing values (`df.isnull().sum()`)

### Step 2: Separate and Categorize
- Separate numerical and categorical columns
- Identify target variable
- Identify ID columns (should be excluded from modeling)

### Step 3: Numerical Feature Inspection
- Generate summary statistics (`df.describe()`)
- Calculate skewness and kurtosis for each feature
- Plot histograms for each numerical feature
- Plot boxplots for each numerical feature
- Identify and document outliers

### Step 4: Categorical Feature Inspection
- Print value counts for each categorical feature
- Identify rare categories (< 1-5% of data)
- Check for inconsistent labeling or data quality issues
- Visualize distributions with bar charts

### Step 5: Target-Feature Relationships
- Compare numerical feature distributions across target classes
- Compare categorical feature frequencies across target classes
- Identify features with strong discriminative power
- Flag features with weak signal

### Step 6: Decision Making
- Document skewness observations and transformation recommendations
- Flag outliers and decide handling strategy
- Identify rare categories requiring grouping
- Decide scaling requirements based on algorithm choice
- Plan feature engineering steps

**This systematic process prevents oversights and ensures reproducibility.**

## Documenting Distribution Observations

Professional ML projects require documentation of distribution insights. This creates **traceability** and **justifies preprocessing decisions** to stakeholders and future reviewers.

### Example Documentation Template

```markdown
## Feature Distribution Analysis

### Numerical Features

**TotalCharges**
- Highly right-skewed (skewness = 1.8)
- Extreme outliers present above 8000
- Recommendation: Apply log transformation
- Rationale: Will improve linear model performance

**Tenure**
- Moderately left-skewed (skewness = -0.4)
- No extreme outliers
- Recommendation: No transformation required
- Rationale: Distribution acceptable for all algorithm families

**MonthlyCharges**
- Approximately symmetric (skewness = 0.1)
- No extreme outliers
- Recommendation: No transformation required

### Categorical Features

**Contract**
- Month-to-month: 55% (dominant)
- One-year: 25%
- Two-year: 20%
- Recommendation: No grouping needed, all categories substantial

**PaymentMethod**
- Electronic check: 35% (most common)
- Bank transfer: 25%
- Mailed check: 20%
- Credit card: 15%
- Other: 5% (rare)
- Recommendation: Consider grouping "Other" with "Mailed check"

### Target Comparison

- Churned customers have **higher median MonthlyCharges** (suggests strong signal)
- Churned customers have **significantly lower tenure** (clear pattern)
- Implications: Both features are likely valuable predictors

### Preprocessing Decisions

1. Apply log transformation to TotalCharges
2. Drop or scale extreme TotalCharges outliers (investigation required)
3. Group PaymentMethod "Other" into "Mailed check"
4. Apply StandardScaler for distance-based models
```

### Why Documentation Matters

- **Reproducibility** — future you (or collaborators) understands your reasoning
- **Stakeholder communication** — explains data preparation to non-technical audiences
- **Model audit trail** — justifies design choices if model performance is questioned
- **Transfer learning** — insights guide preprocessing for similar future projects

## Common Mistakes in Distribution Inspection

### Mistake 1: Skipping Visualization

**The Error:**
Relying only on summary statistics (mean, median, std) and ignoring visual inspection.

**Why it fails:**
Two features can have identical summary statistics but completely different shapes. Anscombe's Quartet is the classic example — four datasets with identical means, variances, and correlation coefficients but very different distributions.

**Fix:**
Always plot histograms and boxplots. Always.

### Mistake 2: Ignoring Outliers

**The Error:**
Seeing outliers in boxplots but deciding they are "probably fine" and proceeding without investigation.

**Why it fails:**
Outliers can dominate loss functions and distort model learning. A single extreme value can shift the mean significantly.

**Fix:**
Investigate outliers: Are they valid (real phenomena) or errors? Document your decision.

### Mistake 3: Transforming Without Justification

**The Error:**
Applying log transforms automatically because "everyone does it" without checking skewness.

**Why it fails:**
- Reduces interpretability unnecessarily
- May not improve performance if data isn't skewed
- Makes model explanation harder for stakeholders

**Fix:**
Transform only when distribution analysis justifies it (e.g., skewness > 1).

### Mistake 4: Ignoring Categorical Imbalance

**The Error:**
Failing to notice that one categorical variable has 95% of one class and 5% of another.

**Why it fails:**
Severe imbalance can distort model learning and evaluation metrics.

**Fix:**
Always inspect categorical frequencies. Plan for imbalance handling if needed.

### Mistake 5: Inspecting After Splitting Incorrectly

**The Error:**
Computing transformation parameters (mean, std) or rare category thresholds using test set data.

**Why it fails:**
This is **data leakage** — your model indirectly learns from test data.

**Fix:**
Always compute statistics on training data only. Apply those statistics to test data.

**Remember:** Inspection should inform decisions made during preprocessing (fitting), not during prediction.

## Practical Application: The Inspection Checklist

Before training your first model in this sprint, use this checklist:

### For Every Numerical Feature
- [ ] Check range (min, max) — any impossible values?
- [ ] Calculate skewness — is it > 0.5, > 1?
- [ ] Plot histogram — symmetric or skewed?
- [ ] Plot boxplot — any outliers?
- [ ] Calculate percentiles (25th, 50th, 75th, 95th)
- [ ] Decision: Transform? Scale? Handle outliers?

### For Every Categorical Feature
- [ ] Count unique categories
- [ ] Print value counts — any imbalance?
- [ ] Identify rare categories (< 1-5% threshold)
- [ ] Check for inconsistent labeling
- [ ] Plot bar chart of frequencies
- [ ] Decision: Group rare categories? Clean labels?

### For Target-Feature Relationships
- [ ] Compare numerical features across target classes
- [ ] Visualize with boxplots or violin plots
- [ ] Compare categorical feature frequencies by target
- [ ] Identify strong vs weak signals
- [ ] Document discriminative features

### Documentation
- [ ] Create summary document of findings
- [ ] Justify transformation decisions
- [ ] Document rare category handling
- [ ] List features with quality issues
- [ ] Identify features with weak predictive signal

**This ensures your model training phase is grounded in real understanding, not blind execution.**

## Closing Reflection

Feature distribution inspection is where **intuition meets evidence**.

### The Core Principle

**You do not model data you do not understand.**

Before tuning hyperparameters, before cross-validation, before optimizing metrics — **inspect. Understand. Document.**

### What Sets Apart Effective ML Practitioners

The best ML practitioners are not those who memorize algorithms. They are those who:

- **Understand their data deeply** before modeling
- **Ask hard questions** about quality, distribution, and relevance
- **Make evidence-based decisions** about preprocessing
- **Document their reasoning** for reproducibility
- **Iterate on understanding** when model performance diverges from expectations

### The Workflow

1. **Inspect first**
2. **Preprocess intentionally** (based on inspection)
3. **Model second**
4. **Evaluate carefully** (compare predictions to data reality)
5. **Iterate** (re-inspect if model underperforms)

### Moving Forward

In your next lesson, you will train models. You will tune hyperparameters. You will optimize metrics.

But only after you have inspected your data thoroughly.

**Inspect first. Model second.**

## Bonus Content 🎁

This section is optional, and learners who want to explore the topics covered in this lesson can utilize the materials provided below.

### Data Visualization Best Practices

**Books:**
- "Exploratory Data Analysis" by John Tukey — foundational
- "The Visual Display of Quantitative Information" by Edward Tufte — visualization principles

**Online Resources:**
- Seaborn documentation: https://seaborn.pydata.org/
- Matplotlib tutorial: https://matplotlib.org/stable/tutorials/index

### Exploratory Data Analysis (EDA) Workflows

**Tools:**
- `pandas-profiling` — automated EDA report generation
- `ydata-profiling` — modern EDA profiling
- `Sweetviz` — visual EDA analysis

**Example:**
```python
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="Data Profile")
profile.to_notebook_iframe()  # Display in Jupyter
```

### Understanding Skewness and Kurtosis

**Skewness:**
- Measures asymmetry of distribution
- Interpretable as: negative (left tail), zero (symmetric), positive (right tail)
- Impacts linear models strongly

**Kurtosis:**
- Measures "tailedness" — presence of extreme values
- High kurtosis → heavy tails (outliers common)
- Low kurtosis → light tails (outliers rare)

```python
df['Feature'].skew()
df['Feature'].kurtosis()
```

### Advanced Transformation Methods

**Yeo-Johnson Transformation:**
- Handles zero and negative values (unlike Box-Cox)
- Automatically optimal parameter selection

```python
from scipy.stats import yeojohnson
df['Feature_yj'], lambda_param = yeojohnson(df['Feature'])
```

**Quantile Transformation:**
- Maps features to uniform or normal distribution
- Robust to outliers

```python
from sklearn.preprocessing import QuantileTransformer
qt = QuantileTransformer(output_distribution='normal')
df_transformed = qt.fit_transform(df[numerical_cols])
```

### Further Learning Topics

- Multivariate feature analysis (correlations, PCA)
- Handling imbalanced datasets
- Feature encoding strategies for categorical data
- Missing value analysis and imputation strategies